# Simple Plan Node Prototype - Step 1: Registry from Sitemap

This notebook starts with the simplest possible registry builder:
1. Load `https://www.fretchen.eu/sitemap.xml`
2. Extract all `<loc>` URLs
3. Normalize URLs (remove query/fragment, stable trailing slash policy)
4. Save a basic registry JSON file

In [23]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
from urllib.request import urlopen
import xml.etree.ElementTree as ET

SITEMAP_URL = "https://www.fretchen.eu/sitemap.xml"
OUT_PATH = Path("../state/simple_planner/registry.json")


def normalize_url(url: str) -> str:
    parts = urlsplit(url.strip())
    path = parts.path or "/"
    if path != "/" and path.endswith("/"):
        path = path[:-1]
    return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), path, "", ""))


with urlopen(SITEMAP_URL) as response:
    raw_xml = response.read()

root = ET.fromstring(raw_xml)
ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
loc_nodes = root.findall("sm:url/sm:loc", ns)

registry_urls = []
seen = set()
for node in loc_nodes:
    normalized = normalize_url(node.text or "")
    if not normalized or normalized in seen:
        continue
    seen.add(normalized)
    registry_urls.append(normalized)

registry = {
    "source": SITEMAP_URL,
    "count": len(registry_urls),
    "urls": registry_urls,
}

print(f"Loaded {registry['count']} URLs from sitemap")
registry_urls[:10]

Loaded 73 URLs from sitemap


['https://www.fretchen.eu/',
 'https://www.fretchen.eu/blog',
 'https://www.fretchen.eu/quantum',
 'https://www.fretchen.eu/blog/0',
 'https://www.fretchen.eu/blog/1',
 'https://www.fretchen.eu/blog/10',
 'https://www.fretchen.eu/blog/11',
 'https://www.fretchen.eu/blog/12',
 'https://www.fretchen.eu/blog/13',
 'https://www.fretchen.eu/blog/14']

In [24]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(json.dumps(registry, indent=2), encoding="utf-8")
print(f"Wrote registry to {OUT_PATH}")

# quick sanity check
print("First 5 registry URLs:")
for url in registry['urls'][:5]:
    print("-", url)

Wrote registry to ../state/simple_planner/registry.json
First 5 registry URLs:
- https://www.fretchen.eu/
- https://www.fretchen.eu/blog
- https://www.fretchen.eu/quantum
- https://www.fretchen.eu/blog/0
- https://www.fretchen.eu/blog/1


Writing the registry for the moment locally for some first tests. However, later it shall be saved with the other files in the S3 bucket.

In [25]:
# Build a cleaned registry by removing hand-curated excluded URLs.
# Expected structure in registry.json:
# {
#   "source": "...",
#   "count": 123,
#   "urls": [...],
#   "excluded": [...]
# }

CLEAN_OUT_PATH = OUT_PATH.with_name("registry_clean.json")

raw_registry = json.loads(OUT_PATH.read_text(encoding="utf-8"))
excluded_raw = raw_registry.get("excluded", [])

excluded_normalized = {normalize_url(u) for u in excluded_raw if isinstance(u, str) and u.strip()}

clean_urls = []
for u in raw_registry.get("urls", []):
    nu = normalize_url(u)
    if nu in excluded_normalized:
        continue
    clean_urls.append(nu)

clean_registry = {
    "source": raw_registry.get("source", SITEMAP_URL),
    "count": len(clean_urls),
    "urls": clean_urls,
    "excluded": sorted(excluded_normalized),
}

CLEAN_OUT_PATH.write_text(json.dumps(clean_registry, indent=2), encoding="utf-8")

print(f"Excluded: {len(excluded_normalized)}")
print(f"Original URLs: {len(raw_registry.get('urls', []))}")
print(f"Clean URLs: {clean_registry['count']}")
print(f"Wrote cleaned registry to {CLEAN_OUT_PATH}")

Excluded: 0
Original URLs: 73
Clean URLs: 73
Wrote cleaned registry to ../state/simple_planner/registry_clean.json


In [26]:
# Read the real queue from S3 and build "latest publication per page" history.
# Uses the same S3 location as scw_js/growth_service.ts.

import os
from datetime import datetime, timezone
from pathlib import Path

import boto3
from dotenv import load_dotenv

S3_BUCKET = "my-imagestore"
S3_REGION = "nl-ams"
S3_ENDPOINT = "https://s3.nl-ams.scw.cloud"
S3_STATE_KEY = "growth-agent/content_queue.json"

# Load environment variables from growth-agent/.env (notebooks run from notebooks/)
dotenv_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
]
for dotenv_path in dotenv_candidates:
    if dotenv_path.exists():
        load_dotenv(dotenv_path=dotenv_path, override=False)
        print(f"Loaded env from {dotenv_path.name}")
        break

access_key = os.environ.get("SCW_ACCESS_KEY")
secret_key = os.environ.get("SCW_SECRET_KEY")
if not access_key or not secret_key:
    raise RuntimeError(
        "Missing SCW_ACCESS_KEY/SCW_SECRET_KEY. Add them to growth-agent/.env or export them in shell."
    )

s3 = boto3.client(
    "s3",
    region_name=S3_REGION,
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

obj = s3.get_object(Bucket=S3_BUCKET, Key=S3_STATE_KEY)
queue = json.loads(obj["Body"].read())

published = queue.get("published", [])
now = datetime.now(timezone.utc)

latest_by_url = {}
for draft in published:
    link = draft.get("link")
    if not link:
        continue

    page_url = normalize_url(link)

    ts_raw = draft.get("scheduled_at") or draft.get("created")
    if not ts_raw:
        continue

    ts = datetime.fromisoformat(ts_raw.replace("Z", "+00:00"))
    if page_url not in latest_by_url or ts > latest_by_url[page_url]:
        latest_by_url[page_url] = ts

publication_history = []
for page_url, last_ts in latest_by_url.items():
    t_days = (now - last_ts).total_seconds() / 86400.0
    publication_history.append(
        {
            "page_url": page_url,
            "last_published_at": last_ts.isoformat(),
            "t_days": round(t_days, 2),
        }
    )

publication_history.sort(key=lambda x: x["t_days"])

print(f"Loaded queue from s3://{S3_BUCKET}/{S3_STATE_KEY}")
print(f"Published drafts in queue: {len(published)}")
print(f"Unique pages with publication history: {len(publication_history)}")
publication_history[:10]

Loaded env from .env
Loaded queue from s3://my-imagestore/growth-agent/content_queue.json
Published drafts in queue: 8
Unique pages with publication history: 3


[{'page_url': 'https://www.fretchen.eu/x402',
  'last_published_at': '2026-04-22T07:00:00+00:00',
  't_days': 1.74},
 {'page_url': 'https://www.fretchen.eu/quantum/amo',
  'last_published_at': '2026-04-20T09:00:00+00:00',
  't_days': 3.66},
 {'page_url': 'https://www.fretchen.eu/blog/25',
  'last_published_at': '2026-04-18T09:00:00+00:00',
  't_days': 5.66}]

In [27]:
# Draw next pages using half-life decay on recently published pages.
# KISS approach: start from clean registry, reduce weight by recency only.

import random
import math
from urllib.parse import urlsplit, urlunsplit

N_SAMPLES = 3
HALF_LIFE_DAYS = 30.0
RANDOM_SEED = 43

# random.seed(RANDOM_SEED)


def canonical_page_url(url: str) -> str:
    p = urlsplit((url or "").strip())
    path = p.path or "/"
    if path != "/" and path.endswith("/"):
        path = path[:-1]
    return urlunsplit((p.scheme.lower(), p.netloc.lower(), path, "", ""))


# Build quick lookup: page -> days since last publication
last_days_by_page = {
    canonical_page_url(item["page_url"]): float(item["t_days"])
    for item in publication_history
}

candidate_urls = [canonical_page_url(u) for u in clean_registry.get("urls", [])]

weighted_candidates = []
for page_url in candidate_urls:
    t_days = last_days_by_page.get(page_url)

    if t_days is None:
        weight = 1.0
    else:
        # Pure half-life recovery: recently used pages get lower weight,
        # then recover smoothly over time.
        weight = 1.0 - math.pow(2.0, -t_days / HALF_LIFE_DAYS)

    weighted_candidates.append(
        {
            "page_url": page_url,
            "t_days": t_days,
            "weight": weight,
        }
    )

# Weighted sample without replacement (simple iterative KISS)
chosen = []
pool = weighted_candidates.copy()
for _ in range(min(N_SAMPLES, len(pool))):
    weights = [c["weight"] for c in pool]
    picked = random.choices(pool, weights=weights, k=1)[0]
    chosen.append(picked)
    pool = [c for c in pool if c["page_url"] != picked["page_url"]]

print(f"Candidates total: {len(candidate_urls)}")
print(f"Sampled pages: {len(chosen)}")
print("\nSelected pages:")
for i, item in enumerate(chosen, start=1):
    print(f"{i}. {item['page_url']} | t_days={item['t_days']} | weight={item['weight']:.4f}")

chosen

Candidates total: 73
Sampled pages: 3

Selected pages:
1. https://www.fretchen.eu/quantum/qml/3 | t_days=None | weight=1.0000
2. https://www.fretchen.eu/imagegen | t_days=None | weight=1.0000
3. https://www.fretchen.eu/blog/7 | t_days=None | weight=1.0000


[{'page_url': 'https://www.fretchen.eu/quantum/qml/3',
  't_days': None,
  'weight': 1.0},
 {'page_url': 'https://www.fretchen.eu/imagegen',
  't_days': None,
  'weight': 1.0},
 {'page_url': 'https://www.fretchen.eu/blog/7', 't_days': None, 'weight': 1.0}]

## Use Real Planner Functions From `plan.py`

This section executes the production planner helpers directly:

1. Step 1: draw candidate pages with `select_pages_for_plan(...)`
2. Step 2: create scheduled plan items with `build_plan_items(...)`
3. End-to-end: run `plan_node(...)` and inspect `content_plan.json`

This keeps the notebook demo aligned with the actual runtime code.

In [28]:
# Execute real planner functions from agent/nodes/plan.py

from datetime import datetime, timedelta, timezone


from agent.models import ContentQueue, Draft
from agent.nodes.plan import (
    PIPELINE_TARGET,
    build_plan_items,
    plan_node,
    select_pages_for_plan,
)

# Minimal in-memory storage compatible with load_model/storage.write usage
class NotebookStorage:
    def __init__(self, initial: dict):
        self.data = dict(initial)

    def read(self, key):
        return self.data.get(key)

    def write(self, key, value):
        if hasattr(value, "model_dump"):
            self.data[key] = value.model_dump(mode="json")
        else:
            self.data[key] = value


# Use cleaned registry from earlier cells and existing queue data if available
registry_payload = {
    "urls": clean_registry.get("urls", []) if "clean_registry" in globals() else []
}

queue_payload = queue if isinstance(globals().get("queue"), dict) else {}
queue_model = ContentQueue.model_validate(queue_payload)

# Keep a partial pipeline so drawing and scheduling are visible in the demo
if not queue_model.drafts and not queue_model.approved:
    queue_model.drafts = [
        Draft(
            id="demo-draft-1",
            channel="mastodon",
            language="en",
            content="already planned",
            scheduled_at=datetime.now(timezone.utc).replace(
                hour=9, minute=0, second=0, microsecond=0
            )
            + timedelta(days=1),
        )
    ]

storage = NotebookStorage(
    {
        "content_queue.json": queue_model.model_dump(mode="json"),
        "simple_planner/registry_clean.json": registry_payload,
    }
)

# Same need calculation as the node, so Step 1 is explicit
now = datetime.now(timezone.utc)
future_approved = [d for d in queue_model.approved if d.scheduled_at and d.scheduled_at > now]
existing = len(queue_model.drafts) + len(future_approved)
needed = max(0, PIPELINE_TARGET - existing)

print(f"existing pipeline: {existing}, needed: {needed}")

# Step 1: draw new items
selected_pages, registry_total, history_pages = select_pages_for_plan(
    storage=storage,
    queue=queue_model,
    needed=needed,
    now=now,
)
print(f"step 1 -> selected: {len(selected_pages)}, registry_total: {registry_total}, history_pages: {history_pages}")

# Step 2: schedule new items
items = build_plan_items(queue=queue_model, selected_pages=selected_pages)
print(f"step 2 -> scheduled items: {len(items)}")
print("first 3 items:")
for item in items[:3]:
    print(f"- {item.channel} | {item.scheduled_at.isoformat()} | {item.page_url}")

# End-to-end node execution (writes content_plan.json)
node_result = plan_node({"storage": storage})
plan_payload = storage.read("content_plan.json") or {}
print("node result:", node_result)
print("persisted content_plan items:", len(plan_payload.get("items", [])))
plan_payload

existing pipeline: 9, needed: 1
step 1 -> selected: 1, registry_total: 73, history_pages: 3
step 2 -> scheduled items: 1
first 3 items:
- bluesky | 2026-05-07T09:00:00+00:00 | https://www.fretchen.eu/quantum/amo/6
node result: {'plan_created': True}
persisted content_plan items: 1


{'items': [{'page_url': 'https://www.fretchen.eu/blog/9',
   'page_title': 'Generating AI images, paying anonymously and little | fretchen.eu',
   'page_description': 'I implement anonymous micro-payments for AI image generation using Ethereum Layer-2. See how to connect blockchain payments with serverless AI for sustainable, privacy-focused services.',
   'reason': 'random draw from registry (unpublished page)',
   'channel': 'bluesky',
   'scheduled_at': '2026-05-07T09:00:00Z'}],
 'diagnostics': {'needed': 1,
  'existing_pipeline': 9,
  'registry_total': 73,
  'history_pages': 3,
  'selected_items': 1,
  'selected_unique_urls': 1,
  'half_life_days': 30.0,
  'channels': 'mastodon,bluesky'}}

## Registry Excluded verwalten (S3) - nur URLs

Diese Zellen lesen und schreiben `registry_excluded.json` direkt im gleichen S3-Namespace wie der Growth-Agent.

Aktuell nutzen wir absichtlich nur eine einfache URL-Liste (keine Prefix-Regeln).

Gewuenschte Exclusions:

- https://www.fretchen.eu/growth
- https://www.fretchen.eu/

Vorgehen:

1. Cell 11 ausfuehren (laden)
2. Cell 12 optional anpassen
3. Cell 13 ausfuehren (in S3 speichern)

In [ ]:
# Cell 11: registry_excluded.json aus S3 laden (nur URLs)

from datetime import datetime, timezone

REGISTRY_EXCLUDED_KEY = "growth-agent/registry_excluded.json"
TARGET_EXCLUDED_URLS = [
    "https://www.fretchen.eu/growth",
    "https://www.fretchen.eu/",
]

if "s3" not in globals():
    raise RuntimeError("S3 client `s3` fehlt. Bitte zuerst die S3-Cell oben ausfuehren.")

try:
    excluded_obj = s3.get_object(Bucket=S3_BUCKET, Key=REGISTRY_EXCLUDED_KEY)
    excluded_config_current = json.loads(excluded_obj["Body"].read())
    print(f"Loaded {REGISTRY_EXCLUDED_KEY}")
except s3.exceptions.NoSuchKey:
    excluded_config_current = {
        "urls": TARGET_EXCLUDED_URLS,
        "updated_at": None,
    }
    print(f"{REGISTRY_EXCLUDED_KEY} not found. Starting with target exclusions.")

loaded_urls = excluded_config_current.get("urls", []) if isinstance(excluded_config_current, dict) else []
excluded_urls_edit = sorted({normalize_url(u) for u in loaded_urls if isinstance(u, str) and u.strip()})

print(f"urls: {len(excluded_urls_edit)}")
excluded_urls_edit

growth-agent/registry_excluded.json not found. Starting with empty config.
urls: 0
prefixes: 0


{'urls': [], 'prefixes': [], 'updated_at': None}

In [ ]:
# Cell 12: URL-Exclusions bearbeiten
# Standard: genau diese zwei URLs behalten.

excluded_urls_edit = [
    "https://www.fretchen.eu/growth",
    "https://www.fretchen.eu/",
]

print("Edit state:")
print("urls:", len(excluded_urls_edit))
{"urls_preview": excluded_urls_edit}

Edit state:
urls: 1
prefixes: 1


{'urls_preview': ['https://www.fretchen.eu/impressum'],
 'prefixes_preview': ['/tag/']}

In [ ]:
# Cell 13: edited registry_excluded.json nach S3 schreiben (nur URLs)

excluded_payload = {
    "urls": sorted({normalize_url(u) for u in excluded_urls_edit if isinstance(u, str) and u.strip()}),
    "updated_at": datetime.now(timezone.utc).isoformat(),
}

s3.put_object(
    Bucket=S3_BUCKET,
    Key=REGISTRY_EXCLUDED_KEY,
    Body=json.dumps(excluded_payload, indent=2).encode("utf-8"),
    ContentType="application/json",
)

print(f"Wrote {REGISTRY_EXCLUDED_KEY}")
print(f"urls: {len(excluded_payload['urls'])}")
excluded_payload